## Анализ вероятности ухода клиента из банка

In [60]:
import numpy as np
import pandas as pd
import joblib
import optuna
import time
import warnings

from itertools import product
from tqdm.auto import tqdm
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier



warnings.filterwarnings("ignore", message="X does not have valid feature names*")
RANDOM_STATE = 42

In [45]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [47]:
X, y = train.drop(columns=['id', 'Churn']), train['Churn'].map({'No': 0, 'Yes': 1})
X_test = test.drop(columns=['id'])

In [48]:
numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

X = preprocessor.fit_transform(X)
y = y.to_numpy()

X_test = preprocessor.transform(X_test)

### Создание Baseline

In [ ]:
class modelsHub:
    def __init__(self):
        self.results = []
        self.models = {}

    def _get_scores(self, model, X):
        if hasattr(model, 'predict_proba'):
            return model.predict_proba(X)[:, 1]
        if hasattr(model, 'decision_function'):
            return model.decision_function(X)
        return None
    
    def load_model(self, model_class, path):
        if model_class is CatBoostClassifier and str(path).endswith('.cbm'):
            model = model_class()
            model.load_model(path)
            return model
        return joblib.load(path)

    def add_loaded_model_to_leaderboard(self, model_name, loaded_model, X_valid, y_valid):
        pred_start = time.perf_counter()
        y_pred = loaded_model.predict(X_valid)
        predict_time_sec = time.perf_counter() - pred_start

        y_score = self._get_scores(loaded_model, X_valid)
        row = {
            'model': model_name,
            'fit_time_sec': 0.0,
            'predict_time_sec': predict_time_sec,
            'accuracy': accuracy_score(y_valid, y_pred),
            'precision': precision_score(y_valid, y_pred, zero_division=0),
            'recall': recall_score(y_valid, y_pred, zero_division=0),
            'f1': f1_score(y_valid, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_valid, y_score),
            'params': loaded_model.get_params() if hasattr(loaded_model, 'get_params') else None
        }
        self.results.append(row)
        self.models[model_name] = loaded_model
        return row, loaded_model

    def choice_best(self, model_class, X_train, y_train, model_name, params, general_params=None, n_splits=5):
        best_params = None
        best_auc = -np.inf
        
        for i in tqdm(range(len(params)), total=len(params), desc=f'{model_name} tuning'):
            params_to_use = {**params[i], **general_params} if general_params else params[i]
            try:
                row = self.fit_predict(
                    model_class,
                    X_train,
                    y_train,
                    model_name=model_name,
                    params=params_to_use,
                    n_splits=n_splits,
                    return_model=False
                )

                if row['roc_auc'] > best_auc:
                    best_params = params_to_use
                    best_auc = row['roc_auc']
            except Exception as e:
                print(f"Error with params {params_to_use}: {e}")

        # обучаем лучшую модель на всём train
        best_model = model_class(**best_params)
        best_model.fit(X_train, y_train)
        self.models[model_name] = best_model

        return best_model, best_params

    def fit_predict(self, model_class, X, y, model_name, params=None, n_splits=5, return_model=True):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

        metrics = {
            'accuracy': [],
            'precision': [],
            'recall': [],
            'f1': [],
            'roc_auc': []
        }
        fit_times = []
        pred_times = []

        for train_idx, valid_idx in skf.split(X, y):
            X_train_fold, X_valid_fold = X[train_idx], X[valid_idx]
            y_train_fold, y_valid_fold = y[train_idx], y[valid_idx]

            model = model_class(**params) if params else model_class()

            fit_start = time.perf_counter()
            model.fit(X_train_fold, y_train_fold)
            fit_times.append(time.perf_counter() - fit_start)

            pred_start = time.perf_counter()
            y_pred = model.predict(X_valid_fold)
            pred_times.append(time.perf_counter() - pred_start)

            y_score = self._get_scores(model, X_valid_fold)
            metrics['accuracy'].append(accuracy_score(y_valid_fold, y_pred))
            metrics['precision'].append(precision_score(y_valid_fold, y_pred, zero_division=0))
            metrics['recall'].append(recall_score(y_valid_fold, y_pred, zero_division=0))
            metrics['f1'].append(f1_score(y_valid_fold, y_pred, zero_division=0))
            metrics['roc_auc'].append(roc_auc_score(y_valid_fold, y_score))

        row = {
            'model': model_name,
            'fit_time_sec': np.mean(fit_times),
            'predict_time_sec': np.mean(pred_times),
            'accuracy': np.mean(metrics['accuracy']),
            'precision': np.mean(metrics['precision']),
            'recall': np.mean(metrics['recall']),
            'f1': np.mean(metrics['f1']),
            'roc_auc': np.nanmean(metrics['roc_auc']),
            'params': params
        }
        self.results.append(row)

        if return_model:
            final_model = model_class(**params) if params else model_class()
            final_model.fit(X, y)
            return row, final_model

        return row

    def leaderboard(self, only_best=False):
        if not self.results:
            return pd.DataFrame()

        df = pd.DataFrame(self.results)

        if only_best:
            def _best_index(g):
                if g['roc_auc'].notna().any():
                    return g['roc_auc'].idxmax()
                return g.index[0]
            best_indices = df.groupby('model', group_keys=False).apply(_best_index).values
            df = df.loc[best_indices].reset_index(drop=True)

        df_sorted = df.sort_values('roc_auc', ascending=False).reset_index(drop=True)
        df['params_str'] = df['params'].apply(lambda x: str(x))

        def highlight_best_roc_auc(row):
            best_roc_auc = df_sorted['roc_auc'].max()
            if pd.notna(row['roc_auc']) and row['roc_auc'] == best_roc_auc:
                return ['background-color: #F0FFF0'] * len(row)
            return [''] * len(row)

        return (
            df_sorted.style
            .format({
                'fit_time_sec': '{:.4f}',
                'predict_time_sec': '{:.4f}',
                'accuracy': '{:.4f}',
                'precision': '{:.4f}',
                'recall': '{:.4f}',
                'f1': '{:.4f}',
                'roc_auc': '{:.4f}',
            })
            .apply(highlight_best_roc_auc, axis=1)
        )

In [29]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

hub = modelsHub()

In [30]:
hub.fit_predict(
    LogisticRegression,
    X_train,
    y_train,
    model_name='LogisticRegression',
)
hub.leaderboard()

,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,LogisticRegression,0.6179,0.0026,0.8545,0.6848,0.6557,0.6699,0.9078,None


### RandomForest

In [32]:
n_estimators = [100, 300]
max_depth = [None, 10, 20]
min_samples_leaf = [1, 2]
max_features = ['sqrt', 'log2']

param_grid = list(product(n_estimators, max_depth, min_samples_leaf, max_features))

random_forest, best_params = hub.choice_best(
    model_class=RandomForestClassifier,
    X_train=X_train,
    y_train=y_train,
    model_name='RandomForestClassifier',
    params=[
        {
            'n_estimators': n,
            'max_depth': d,
            'min_samples_leaf': l,
            'max_features': f
        }
        for n, d, l, f in param_grid
    ],
    general_params={
        'random_state': RANDOM_STATE,
        'n_jobs': -1
    },
    n_splits=3
)

random_forest = joblib.load("./saved models/random_forest.joblib")

hub.leaderboard(only_best=True)

RandomForestClassifier tuning: 100%|██████████| 1/1 [00:18<00:00, 18.33s/it]


,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,LogisticRegression,0.6179,0.0026,0.8545,0.6848,0.6557,0.6699,0.9078,None
1,RandomForestClassifier,4.7980,0.5981,0.8414,0.6666,0.5916,0.6269,0.8911,"{'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}"


### CatBoost

In [ ]:
depth = [6, 8, 10]
iterations = [100, 600, 5000]
reg_lambdas = [0.1, 0.5]

param_grid = list(product(depth, iterations, reg_lambdas))
catboost, best_params = hub.choice_best(
    model_class=CatBoostClassifier,
    X_train=X_train,
    y_train=y_train,
    model_name='CatBoostClassifier',
    params=[{'depth': d, 'iterations': it, 'reg_lambda': rl} for d, it, rl in param_grid],
    general_params={'random_state': RANDOM_STATE, 'verbose': 0, 'early_stopping_rounds': 50},
    eval=(X_valid, y_valid)
)

catboost.save_model("./saved models/catboost_model.cbm")

hub.leaderboard(only_best=True)

  0%|          | 0/1 [00:00<?, ?it/s]

Processing CatBoostClassifier...


100%|██████████| 1/1 [00:00<00:00,  3.11it/s]


,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,CatBoostClassifier,0.0000,0.1581,0.8613,0.7134,0.6422,0.6760,0.9166,"{'depth': 6, 'random_seed': 42, 'od_wait': 50, 'l2_leaf_reg': 0.5, 'iterations': 5000, 'verbose': 0, 'loss_function': 'Logloss'}"
1,LogisticRegression,0.6179,0.0026,0.8545,0.6848,0.6557,0.6699,0.9078,None
2,RandomForestClassifier,4.7980,0.5981,0.8414,0.6666,0.5916,0.6269,0.8911,"{'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}"


In [86]:
hub.leaderboard(only_best=True)

,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,CatBoostClassifier,95.3049,0.1411,0.8613,0.7130,0.6425,0.6759,0.9157,"{'depth': 6, 'iterations': 5000, 'reg_lambda': 0.5, 'random_state': 42, 'verbose': 0, 'early_stopping_rounds': 50}"
1,RandomForestClassifier,11.7599,0.4754,0.8567,0.7016,0.6329,0.6655,0.9115,"{'n_estimators': 300, 'max_depth': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}"
2,LogisticRegression,0.6574,0.0021,0.8545,0.6848,0.6557,0.6699,0.9078,None


### LightGBM

In [ ]:
num_leaves = [31, 63]
n_estimators = [100, 500]
learning_rates = [0.05, 0.1]
max_depths = [6, 10, -1]
reg_alphas = [0, 1]
reg_lambdas = [0, 1]

param_grid = list(product(num_leaves, n_estimators, learning_rates, max_depths, reg_alphas, reg_lambdas))
params = [
    {
        'num_leaves': nl,
        'n_estimators': ne,
        'learning_rate': lr,
        'max_depth': md,
        'reg_alpha': ra,
        'reg_lambda': rl
    }
    for nl, ne, lr, md, ra, rl in param_grid
]

lgbm_model, best_params = hub.choice_best(
    model_class=LGBMClassifier,
    X_train=X_train,
    y_train=y_train,
    model_name='LightGBM',
    params=params,
    general_params={'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': -1},
    n_splits=3
)

joblib.dump(lgbm_model, "./saved models/lightgbm.joblib")
hub.leaderboard(only_best=True)

,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,CatBoostClassifier,0.0000,0.1581,0.8613,0.7134,0.6422,0.6760,0.9166,"{'depth': 6, 'random_seed': 42, 'od_wait': 50, 'l2_leaf_reg': 0.5, 'iterations': 5000, 'verbose': 0, 'loss_function': 'Logloss'}"
1,LightGBM,0.0000,0.3605,0.8608,0.7098,0.6460,0.6764,0.9166,"{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': 6, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 500, 'n_jobs': -1, 'num_leaves': 31, 'objective': None, 'random_state': 42, 'reg_alpha': 1, 'reg_lambda': 1, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbosity': -1}"
2,LogisticRegression,0.6179,0.0026,0.8545,0.6848,0.6557,0.6699,0.9078,None
3,RandomForestClassifier,4.7980,0.5981,0.8414,0.6666,0.5916,0.6269,0.8911,"{'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}"


### XGboost

Здесь проведен переход от подбора гиперпараметров по сетке к подбору параметров через optuna. Сделано это для оптимизации и более точного подбора параметров

### Формирование ответа для kaggle

In [49]:
model = hub.models['LightGBM']

test_proba = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'id': test['id'],
    'Churn': test_proba
})

submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.063323
1,594195,0.000767
2,594196,0.108511
3,594197,0.003632
4,594198,0.532360


### Stacking с Logistic Regression как мета-моделью

In [ ]:
def get_oof_predictions(model, X, y, X_test, n_splits=5, random_state=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    oof_train = np.zeros(len(X))
    oof_test = np.zeros(len(X_test))
    test_fold_preds = []

    for train_idx, valid_idx in tqdm(skf.split(X, y), total=n_splits):
        X_train_fold, X_valid_fold = X[train_idx], X[valid_idx]
        y_train_fold = y[train_idx]

        fold_model = clone(model)
        fold_model.fit(X_train_fold, y_train_fold)

        oof_train[valid_idx] = fold_model.predict_proba(X_valid_fold)[:, 1]
        test_fold_preds.append(fold_model.predict_proba(X_test)[:, 1])

    oof_test = np.mean(test_fold_preds, axis=0)
    return oof_train.reshape(-1, 1), oof_test.reshape(-1, 1)

lgbm_oof_train, lgbm_oof_test = get_oof_predictions(lgbm_base, X_train, y_train, X_test, n_splits=3, random_state=RANDOM_STATE)
cat_oof_train, cat_oof_test = get_oof_predictions(catboost, X_train, y_train, X_test, n_splits=3, random_state=RANDOM_STATE)
random_forest_oof_train, random_forest_oof_test = get_oof_predictions(random_forest, X_train, y_train, X_test, n_splits=3, random_state=RANDOM_STATE)

X_stack_train = np.hstack([lgbm_oof_train, cat_oof_train, random_forest_oof_train])
X_stack_test = np.hstack([lgbm_oof_test, cat_oof_test, random_forest_oof_test])

C_values = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]
l1_ratios = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
class_weights = [None, "balanced"]

valid_params = [
    {
        "C": C,
        "l1_ratio": l1r,
        "class_weight": cw
    }
    for C, l1r, cw in product(C_values, l1_ratios, class_weights)
]

stack_logreg, best_params = hub.choice_best(
    model_class=LogisticRegression,
    X_train=X_stack_train,
    y_train=y_train,
    model_name="StackingLogReg",
    params=valid_params,
    general_params={"random_state": RANDOM_STATE, "max_iter": 4000, "solver": "saga"},
    n_splits=5
)

hub.leaderboard(only_best=True)

StackingLogReg tuning: 100%|██████████| 154/154 [04:55<00:00,  1.92s/it]


,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,CatBoostClassifier,0.0000,0.1581,0.8613,0.7134,0.6422,0.6760,0.9166,"{'depth': 6, 'random_seed': 42, 'od_wait': 50, 'l2_leaf_reg': 0.5, 'iterations': 5000, 'verbose': 0, 'loss_function': 'Logloss'}"
1,LightGBM,0.0000,0.3605,0.8608,0.7098,0.6460,0.6764,0.9166,"{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': 6, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 500, 'n_jobs': -1, 'num_leaves': 31, 'objective': None, 'random_state': 42, 'reg_alpha': 1, 'reg_lambda': 1, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbosity': -1}"
2,StackingLogReg,0.3169,0.0005,0.8614,0.7238,0.6219,0.6690,0.9161,"{'C': 0.01, 'solver': 'saga', 'l1_ratio': 0.5, 'class_weight': None, 'random_state': 42, 'max_iter': 4000}"
3,LogisticRegression,0.6179,0.0026,0.8545,0.6848,0.6557,0.6699,0.9078,None
4,RandomForestClassifier,4.7980,0.5981,0.8414,0.6666,0.5916,0.6269,0.8911,"{'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}"


In [58]:
test_proba = stack_logreg.predict_proba(X_stack_test)[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "Churn": test_proba
})

submission.to_csv("submission.csv", index=False)
submission.head()

,id,Churn
0,594194,0.055689
1,594195,0.037171
2,594196,0.067508
3,594197,0.037854
4,594198,0.517419
